[Reference](https://medium.com/@chandravanshi.pankaj.ai/96cf40dbb38d)

# Step 1: Importing all necessary libraries


In [2]:
# === Standard Library ===
import getpass
import os
from typing_extensions import Literal

# === Display Utilities ===
from IPython.display import Image, display
# === Data Modeling ===
from pydantic import BaseModel, Field
# === Formating tool ===
from utils import format_messages
# === LangChain / Tools ===
from langchain_core.messages import SystemMessage, ToolMessage, HumanMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_tavily import TavilySearch
# === LangGraph ===
from langgraph.graph import END, START, StateGraph, MessagesState
# === Load env variables ===
from dotenv import load_dotenv
load_dotenv()

# Step 2: Agent State with Scratchpad

In [4]:
class ScratchpadState(MessagesState):
    """Agent state with an additional scratchpad field for saving intermediate notes."""
    scratchpad: str = Field(description="The scratchpad for storing notes")

# Step 3: Scratchpad Tools


In [5]:
@tool
class WriteToScratchpad(BaseModel):
    """Tool to write notes into the scratchpad memory."""
    notes: str = Field(description="Notes to save to the scratchpad")

@tool
class ReadFromScratchpad(BaseModel):
    """Tool to read previously saved notes from the scratchpad."""
    reasoning: str = Field(description="Why the agent wants to retrieve past notes")

# Step 4: Search Tool and Gemini LLM Setup

In [6]:
#Tavily for real-time web search
search_tool = TavilySearch(max_results=5, topic="general")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=1)

# Step 5: Bind Tools to LLM

In [7]:
tools = [ReadFromScratchpad, WriteToScratchpad, search_tool]
tools_by_name = {tool.name: tool for tool in tools}
llm_with_tools = llm.bind_tools(tools)

# Step 6: System Prompt

In [8]:
scratchpad_prompt = """You are a sophisticated research assistant with access to web search and a persistent scratchpad for note-taking. Your Research Workflow:
1. Check Scratchpad: Look for existing notes relevant to your task.
2. Create Research Plan: Think through the task and outline a plan.
3. Write to Scratchpad: Save the plan and findings.
4. Use Search: Look up information using search tools.
5. Update Scratchpad: Add new results to your notes.
6. Iterate: Repeat as needed.
7. Complete Task: Present final output using all gathered information.
Available Tools:
- WriteToScratchpad
- ReadFromScratchpad
- TavilySearch
"""

# Step 7: Create langgraph workflow

In [9]:
def llm_call(state: ScratchpadState) -> dict:
    return {
        "messages": [
            llm_with_tools.invoke(
                [SystemMessage(content=scratchpad_prompt)] + state["messages"]
            )
        ]
    }

def tool_node(state: ScratchpadState) -> dict:
    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        if tool_call["name"] == "WriteToScratchpad":
            notes = observation.notes
            result.append(ToolMessage(content=f"Wrote to scratchpad: {notes}", tool_call_id=tool_call["id"]))
            update = {"messages": result, "scratchpad": notes}
        elif tool_call["name"] == "ReadFromScratchpad":
            notes = state.get("scratchpad", "")
            result.append(ToolMessage(content=f"Notes from scratchpad: {notes}", tool_call_id=tool_call["id"]))
            update = {"messages": result}
        elif tool_call["name"] == "tavily_search":
            result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
            update = {"messages": result}
    return update
def should_continue(state: ScratchpadState) -> Literal["tool_node", "__end__"]:
    last_message = state["messages"][-1]
    return "tool_node" if last_message.tool_calls else END
agent_builder = StateGraph(ScratchpadState)
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)
agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges("llm_call", should_continue, {"tool_node": "tool_node", END: END})
agent_builder.add_edge("tool_node", "llm_call")
agent = agent_builder.compile()

# Step 8: Display Agent Graph

In [10]:
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

# Step 9: Ask a Query

In [11]:
query = "Compare the funding rounds and recent developments of Xaira vs Cohere."
state = agent.invoke({"messages": [HumanMessage(content=query)]})
format_messages(state['messages'])

# Step 10: Print notes from the scratchpad

In [12]:
from rich.console import Console
from rich.pretty import pprint
console = Console()
console.print("\n[bold green]Scratchpad:[/bold green]")
from rich.markdown import Markdown
Markdown(state['scratchpad'])

# Step 11: In-Memory Store and Namespace Setup

In [13]:
from langgraph.store.memory import InMemoryStore

# Initialize in-memory store for long-term memory
store = InMemoryStore()
# Define a namespace to organize context
namespace = ("rlm", "scratchpad")
# Add persistent context to the store
store.put(
    namespace,
    "scratchpad",
    {
        "scratchpad": "Research project on renewable energy adoption in developing countries. Key areas to track: policy frameworks, technology barriers, financing mechanisms, and success stories from pilot programs."
    }
)

# Step 12: View Stored Memory

In [14]:
from rich.console import Console
from pprint import pprint

# Retrieve the stored scratchpad data
scratchpad = store.get(namespace, "scratchpad")
# Display the stored data
console = Console()
console.print("\n[bold green]Retrieved Context from Memory:[/bold green]")
pprint(scratchpad)

# Step 13: Persistent Tool Node with Store

In [15]:
from langgraph.store.base import BaseStore
from langgraph.checkpoint.memory import InMemorySaver

def tool_node_persistent(state: ScratchpadState, store: BaseStore) -> dict:
    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        if tool_call["name"] == "WriteToScratchpad":
            notes = observation.notes
            result.append(ToolMessage(content=f"Wrote to scratchpad: {notes}", tool_call_id=tool_call["id"]))
            store.put(namespace, "scratchpad", {"scratchpad": notes})
            update = {"messages": result}
        elif tool_call["name"] == "ReadFromScratchpad":
            stored_data = store.get(namespace, "scratchpad")
            notes = stored_data.value["scratchpad"] if stored_data else "No notes found"
            result.append(ToolMessage(content=f"Notes from scratchpad: {notes}", tool_call_id=tool_call["id"]))
            update = {"messages": result}
        elif tool_call["name"] == "tavily_search":
            result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
            update = {"messages": result}
    return update

# Step 14: Build and Compile Agent with Persistent Memory and Checkpointer

In [16]:
agent_builder_persistent = StateGraph(ScratchpadState)
agent_builder_persistent.add_node("llm_call", llm_call)
agent_builder_persistent.add_node("tool_node", tool_node_persistent)
agent_builder_persistent.add_edge(START, "llm_call")
agent_builder_persistent.add_conditional_edges("llm_call", should_continue, {"tool_node": "tool_node", END: END})
agent_builder_persistent.add_edge("tool_node", "llm_call")

# Checkpoint for thread history
checkpointer = InMemorySaver()
# Memory store for long-term context
memory_store = InMemoryStore()
# Compile persistent agent
agent = agent_builder_persistent.compile(
    checkpointer=checkpointer,
    store=memory_store
)

# Step 15: Invoke Agent with Thread ID and Display Output


In [17]:
config = {"configurable": {"thread_id": "1"}}
state = agent.invoke({
    "messages": [HumanMessage(content="Can you search for funding rounds and recent developments of Commonwealth Fusion Systems?")]
}, config)

console.print("\n[bold cyan]Workflow Result (Thread 1) - Kick Off Research:[/bold cyan]")
format_messages(state['messages'])

# Step 16: Access Scratchpad From Previous Conversation With New Thread


In [18]:
# Cross-thread memory persistence demonstration
config_2 = {"configurable": {"thread_id": "2"}}
messages_2 = agent.invoke({
    "messages": [HumanMessage(content="How does the funding raised for Helion Energy compare to Commonwealth Fusion Systems?")]
}, config_2)

console.print("\n[bold cyan]Workflow Result (Thread 2) - Cross-Thread Memory Access:[/bold cyan]")
format_messages(messages_2['messages'])